# 03 — Shapefile Join & Enrichment
**Project:** Supervised Classification of Agricultural Soil Types in Togo  
**Author:** Daniel ESSONANI | Supervised by [M. Leri TCHANTCHO](https://www.linkedin.com/in/leri-damigouri-tchantcho-28503873/)

---

## Overview

This notebook merges three data sources into a single enriched GeoDataFrame:

1. **Cantons shapefile** — 599 polygons with administrative attributes
2. **SoilGrids extraction** (`sols_togo_cantons.csv`) — WRB soil type per canton
3. **Agronomic reference table** (`table_reference_sols.csv`) — fertility, crops, constraints

The result is exported as `cantons_togo_sols.shp`, ready for QGIS styling and qgis2web export.

---

## ⚠️ Known Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|---------|
| Shapefile column names truncated to 10 chars | ESRI Shapefile format limitation | Renamed all columns to ≤10 char ASCII names before export |
| Some cantons had `NaN` in soil column after join | A few OBJECTID mismatches between CSV and shapefile | Used `how='left'` to keep all cantons; NaN filled with `"Unknown"` |
| `ValueError: Cannot convert NaN to integer` during QGIS import | Mixed types in numeric columns | Explicitly cast all numeric columns before export |


## 1. Imports

In [ ]:
import geopandas as gpd
import pandas as pd

print("Libraries loaded ✅")


## 2. Load All Three Sources

In [ ]:
# ── Paths — adjust if needed ──────────────────────────────────────────────────
SHAPEFILE_PATH    = "data/cantons_togo.shp"
SOILGRIDS_CSV     = "sols_togo_cantons.csv"
REFERENCE_CSV     = "table_reference_sols.csv"

# Load
cantons     = gpd.read_file(SHAPEFILE_PATH)
df_sols     = pd.read_csv(SOILGRIDS_CSV)
df_ref      = pd.read_csv(REFERENCE_CSV)

print(f"Cantons loaded:    {len(cantons)} rows")
print(f"SoilGrids data:    {len(df_sols)} rows")
print(f"Reference table:   {len(df_ref)} rows")
print(f"\nCantons CRS: {cantons.crs}")


## 3. Join 1 — Cantons + SoilGrids Results

Join on `OBJECTID` (unique canton identifier present in both datasets).  
`how='left'` ensures all 599 cantons are kept even if a soil value is missing.


In [ ]:
# ── Join 1: cantons ← SoilGrids ───────────────────────────────────────────────
cantons_enriched = cantons.merge(
    df_sols[["OBJECTID", "soil_dominant", "soil_prob1", "soil_prob1_pct",
             "soil_prob2", "soil_prob2_pct"]],
    on="OBJECTID",
    how="left"
)

# Fill missing soil values
cantons_enriched["soil_dominant"] = cantons_enriched["soil_dominant"].fillna("Unknown")

missing_count = (cantons_enriched["soil_dominant"] == "Unknown").sum()
print(f"Cantons with missing soil data: {missing_count}")
print(cantons_enriched[["CANTON", "soil_dominant"]].head(5))


## 4. Join 2 — Add Agronomic Attributes

Join on `soil_dominant` (WRB class name) to attach fertility, crops and constraints.


In [ ]:
# ── Join 2: + agronomic reference ────────────────────────────────────────────
cantons_enriched = cantons_enriched.merge(
    df_ref,
    left_on="soil_dominant",
    right_on="soil",
    how="left"
)

# Drop redundant 'soil' column from reference table
cantons_enriched.drop(columns=["soil"], inplace=True, errors="ignore")

# Fill NaN for cantons with unknown soil
for col in ["fertility", "crops", "constraints"]:
    cantons_enriched[col] = cantons_enriched[col].fillna("No data")

print("\nFull enriched dataset sample:")
print(cantons_enriched[["CANTON", "PREFECTURE", "soil_dominant",
                          "fertility", "crops", "constraints"]].head(8).to_string())


## 5. Rename Columns for Shapefile Compatibility

ESRI Shapefile format truncates column names to **10 characters**.  
We rename all columns to short, meaningful ASCII names to avoid data loss.


In [ ]:
# ── Column renaming map ───────────────────────────────────────────────────────
rename_map = {
    "OBJECTID":      "OBJECTID",
    "CODE_REGIO":    "COD_REG",
    "REGION":        "REGION",
    "CODE_PREFE":    "COD_PRF",
    "PREFECTURE":    "PREFECT",
    "CODE_COMMU":    "COD_COM",
    "COMMUNE":       "COMMUNE",
    "CODE_CANTO":    "COD_CAN",
    "CANTON":        "CANTON",
    "soil_dominant": "SOIL",
    "soil_prob1":    "SOIL_P1",
    "soil_prob1_pct":"P1_PCT",
    "soil_prob2":    "SOIL_P2",
    "soil_prob2_pct":"P2_PCT",
    "fertility":     "FERTILITY",
    "crops":         "CROPS",
    "constraints":   "CONSTRNTS",
}

# Keep only columns in the rename map + geometry
cols_to_keep = [c for c in rename_map if c in cantons_enriched.columns] + ["geometry"]
cantons_final = cantons_enriched[cols_to_keep].rename(columns=rename_map)

print("Final columns:")
for col in cantons_final.columns:
    if col != "geometry":
        print(f"  {col:<12} ({len(col)} chars)")


## 6. Export Enriched Shapefile

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
OUTPUT_SHP = "cantons_togo_sols.shp"

cantons_final.to_file(OUTPUT_SHP, driver="ESRI Shapefile", encoding="utf-8")

print(f"✅ Enriched shapefile saved: {OUTPUT_SHP}")
print(f"   {len(cantons_final)} cantons | {len(cantons_final.columns)-1} attributes + geometry")

# Quick validation
test = gpd.read_file(OUTPUT_SHP)
print(f"\nValidation read-back: {len(test)} features ✅")
test[["CANTON", "SOIL", "FERTILITY", "CROPS"]].head(5)
